# NanoBodyBuilder2 on the same 597 nanobodies

Head-to-head comparator for the frozen SimpleFold-100M run. Same targets, same scoring code,
same schema (`METRICS.md`), so the two are directly mergeable.

**The one real hazard here:** NanoBodyBuilder2 emits **IMGT-numbered** PDBs, not residues numbered
1..L. Mapping its output onto `hseq` by index arithmetic would silently misalign every residue and
still produce believable RMSDs. Every structure is therefore mapped by **sequence identity**, and a
target is dropped rather than guessed at if the sequence does not match.

In [ ]:
import os, sys, json, time, glob, shutil, subprocess, collections
import numpy as np

WORK, SCRATCH = '/kaggle/working', '/tmp/nbb'
os.makedirs(SCRATCH, exist_ok=True)
PRED = f'{SCRATCH}/pred'; os.makedirs(PRED, exist_ok=True)
T0 = time.time()

R = {'model': 'NanoBodyBuilder2',
     'settings': {'numbering_scheme': 'imgt', 'refinement': 'openmm (ImmuneBuilder default)'},
     'metric': 'ImmuneBuilder protocol: IMGT regions, N/CA/C/CB atoms',
     'reference_only': {'SimpleFold_100M_this_benchmark':
                        {'cdr1': 1.955, 'cdr2': 1.241, 'cdr3': 2.942, 'framework': 0.826},
                        'note': 'SAME 597 targets, same scoring code -- this IS a paired comparison'}}

def note(tag):
    R['elapsed_min'] = round((time.time() - T0) / 60, 1)
    R['stage'] = tag
    json.dump(R, open(f'{WORK}/results_nbb2.json', 'w'), indent=2, default=str)
    print(f"[{R['elapsed_min']:>6.1f} min] {tag}", flush=True)

import torch
R['gpus'] = [torch.cuda.get_device_name(g) for g in range(torch.cuda.device_count())]
print('GPUs visible:', R['gpus'] or 'none')

_ds = glob.glob('/kaggle/input/**/targets_vhh.json', recursive=True)
assert len(_ds) == 1, _ds
TG = json.load(open(_ds[0]))
for t in TG:
    assert t['atoms'] == ['N','CA','C','CB'] and t['numbering'] == 'IMGT'
    for k, col in (('cdr1','CDRH1'), ('cdr2','CDRH2'), ('cdr3','CDRH3')):
        assert t['hseq'][t[k][0]:t[k][1]] == t[col], (t['instance'], k)
assert len({t['instance'] for t in TG}) == len(TG)
R['n_targets'] = len(TG)

# ImmuneBuilder asserts min(IMGT number) < 8 and max > 120, i.e. it refuses chains whose
# N-terminus was not resolved. IMGT CDR1 starts at 27, so the first residue is ~27-cdr1[0].
# SimpleFold folded these happily; NanoBodyBuilder2 will not, so expect a handful of refusals.
for t in TG:
    t['_first_imgt'] = 27 - t['cdr1'][0]
R['expect_reject'] = [f"{t['pdb']}{t['H']}" for t in TG if t['_first_imgt'] >= 8]
PROBE = next(t for t in TG if t['_first_imgt'] < 8)
print(f'{len(TG)} targets validated; '
      f'{len(R["expect_reject"])} likely refused by ImmuneBuilder (truncated N-terminus)')
print('probe target:', PROBE['pdb'] + PROBE['H'])
note('start')

## Install ImmuneBuilder + OpenMM refinement

In [ ]:
def sh(c):
    return subprocess.run(c, shell=True, text=True, capture_output=True)

def must(c, what):
    p = sh(c)
    if p.returncode: raise RuntimeError(f'{what} failed:\n{(p.stdout+p.stderr)[-2000:]}')
    return p

must(f'{sys.executable} -m pip install -q ImmuneBuilder', 'install ImmuneBuilder')
must(f'{sys.executable} -m pip install -q openmm pdbfixer biotite', 'install openmm/pdbfixer/biotite')

# ImmuneBuilder imports ANARCI for numbering, but does NOT declare it as a pip dependency --
# ANARCI is a conda/GitHub package and needs the HMMER binary on PATH.
sh('apt-get -qq update && apt-get -qq install -y hmmer')
R['hmmer'] = sh('which hmmscan').stdout.strip() or 'MISSING'
print('hmmscan:', R['hmmer'])

tried = []
for cmd in (f'{sys.executable} -m pip install -q anarci',
            f'{sys.executable} -m pip install -q git+https://github.com/oxpig/ANARCI.git'):
    p = sh(cmd)
    ok = sh(f'{sys.executable} -c "import anarci"').returncode == 0
    tried.append(f"{cmd.split()[-1]} -> rc={p.returncode} import={'ok' if ok else 'no'}")
    if ok: break
R['anarci_install'] = tried
print('\n'.join(tried))

# prove the whole chain works on ONE real sequence before committing to 597
probe = f'''
from ImmuneBuilder import NanoBodyBuilder2
import openmm, anarci
nb = NanoBodyBuilder2()
nb.predict({{"H": "{PROBE['hseq']}"}}).save("/tmp/probe.pdb")
print("PROBE_OK", openmm.__version__)
'''
chk = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True)
if chk.returncode or 'PROBE_OK' not in chk.stdout:
    raise RuntimeError('NanoBodyBuilder2 end-to-end probe failed:\n'
                       + (chk.stdout + chk.stderr)[-2500:])
R['openmm_version'] = chk.stdout.split('PROBE_OK')[1].strip()
n_probe = sum(1 for l in open('/tmp/probe.pdb') if l.startswith('ATOM'))
R['probe_atoms'] = n_probe
print(f'probe OK: openmm {R["openmm_version"]}, {n_probe} atoms written')
note('installed')

## Warm the weights single-threaded, then shard across 2 workers

In [ ]:
# both workers would otherwise race to download the same weights and corrupt them
w = subprocess.run([sys.executable, '-c',
    "from ImmuneBuilder import NanoBodyBuilder2; NanoBodyBuilder2()"],
    capture_output=True, text=True)
if w.returncode:
    raise RuntimeError('weight warm-up failed:\n' + (w.stdout + w.stderr)[-2000:])
print('weights warm')

WORKER = '\n'.join([
 "import sys, json, os, time",
 "from ImmuneBuilder import NanoBodyBuilder2",
 "shard, nshard, tgt, out = int(sys.argv[1]), int(sys.argv[2]), sys.argv[3], sys.argv[4]",
 "TG = json.load(open(tgt))",
 "nb = NanoBodyBuilder2()",
 "t0 = time.time(); n = 0",
 "for i, t in enumerate(TG):",
 "    if i % nshard != shard: continue",
 "    tid = t['pdb'] + t['H']",
 "    p = out + '/' + tid + '.pdb'",
 "    if os.path.exists(p): continue",
 "    try:",
 "        nb.predict({'H': t['hseq']}).save(p)",
 "        n += 1",
 "        if n % 25 == 0: print('shard %d: %d done, %.0fs' % (shard, n, time.time()-t0), flush=True)",
 "    except Exception as e:",
 "        print('FAIL %s %s: %s' % (tid, type(e).__name__, str(e)[:120]), flush=True)",
 "print('shard %d finished %d in %.0fs' % (shard, n, time.time()-t0), flush=True)"])
open(f'{SCRATCH}/worker.py','w').write(WORKER)
json.dump(TG, open(f'{SCRATCH}/tg.json','w'))

t0 = time.time(); procs = []
for g in (0, 1):
    procs.append(subprocess.Popen([sys.executable, f'{SCRATCH}/worker.py', str(g), '2',
                                   f'{SCRATCH}/tg.json', PRED],
                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                 env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)}))
    print(f'launched worker {g}', flush=True)
for g, p in enumerate(procs):
    out = p.communicate()[0]
    R.setdefault('worker_rc', {})[f'w{g}'] = p.returncode
    tail = [l for l in out.split('\n') if 'FAIL' in l or 'finished' in l]
    print(f'--- worker {g} (rc={p.returncode}) ---'); print('\n'.join(tail[-8:]))
    R.setdefault('worker_fail', {})[f'w{g}'] = [l for l in tail if 'FAIL' in l][:20]
R['build_sec'] = round(time.time() - t0, 1)
R['n_predicted'] = len(glob.glob(f'{PRED}/*.pdb'))
R['sec_per_target'] = round(R['build_sec'] / max(R['n_predicted'], 1), 2)
print(f"\n{R['n_predicted']}/{len(TG)} built in {R['build_sec']/60:.1f} min "
      f"({R['sec_per_target']}s/target)")
note('built')

## Score — identical code to the SimpleFold run, plus a sequence-identity guard

NanoBodyBuilder2 numbers residues by IMGT, so its `res_id` is **not** `1..L`. The prediction is
mapped onto `hseq` by reading its actual sequence and locating it, never by index arithmetic.

In [ ]:
import biotite.structure.io.pdb as pdb_io

AA3to1 = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G',
          'HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S',
          'THR':'T','TRP':'W','TYR':'Y','VAL':'V'}
REGIONS = ['cdr1','cdr2','cdr3','framework']

def kabsch(P, Q):
    Pm, Qm = P.mean(0), Q.mean(0)
    V, _, W = np.linalg.svd((P - Pm).T @ (Q - Qm))
    D = np.diag([1, 1, np.sign(np.linalg.det(V @ W))])
    return V @ D @ W, Pm, Qm

rmsd = lambda a, b: float(np.sqrt(((a - b) ** 2).sum(-1).mean()))

rows = []
for t in TG:
    tid = t['pdb'] + t['H']
    try:
        f = f'{PRED}/{tid}.pdb'
        if not os.path.exists(f):
            R.setdefault('missing', []).append(tid); continue
        a = pdb_io.PDBFile.read(f).get_structure(model=1)
        a = a[np.isin(a.atom_name, ['N','CA','C','CB'])]

        seen, order = {}, []
        for ch, ri, ins, rn in zip(a.chain_id, a.res_id, a.ins_code, a.res_name):
            key = (str(ch), int(ri), str(ins))
            if key not in seen:
                seen[key] = len(order); order.append(AA3to1.get(str(rn), 'X'))
        pseq = ''.join(order)

        off = t['hseq'].find(pseq)
        if off < 0:
            R.setdefault('seq_mismatch', []).append(
                f"{tid}: pred {len(pseq)} vs ref {len(t['hseq'])}"); continue
        Ln = len(t['hseq'])
        pred = np.full((Ln, 4, 3), np.nan)
        ai = {'N':0,'CA':1,'C':2,'CB':3}
        for ch, ri, ins, an, xyz in zip(a.chain_id, a.res_id, a.ins_code, a.atom_name, a.coord):
            pred[seen[(str(ch), int(ri), str(ins))] + off, ai[str(an)]] = xyz
        ref = np.array([[[np.nan]*3 if v is None else v for v in r] for r in t['hca']], float)

        reg = {}
        for k in ('cdr1','cdr2','cdr3'):
            m = np.zeros(Ln, bool); lo, hi = t[k]; m[lo:hi] = True; reg[k] = m
        reg['framework'] = ~(reg['cdr1'] | reg['cdr2'] | reg['cdr3'])

        ok = np.isfinite(ref[..., 0]) & np.isfinite(pred[..., 0])
        if ok[:, 1].sum() < 60:
            R.setdefault('skipped', []).append(f'{tid}:coverage'); continue

        row = {'instance': t['instance'], 'tid': tid, 'L': Ln,
               'cdr3_len': int(t['cdr3'][1] - t['cdr3'][0]), 'n_atoms_matched': int(ok.sum()),
               'seq_offset': int(off), 'pred_len': len(pseq),
               'method': t['method'], 'holo': t['holo'], 'ab_split': t['ab_split'],
               'resolution': t['resolution']}
        for mode, fitmask in (('chain', np.ones(Ln, bool)), ('fw', reg['framework'])):
            sel = ok & fitmask[:, None]
            if sel.sum() < 60: continue
            Rot, Pm, Qm = kabsch(pred[sel], ref[sel])
            al = np.full_like(pred, np.nan)
            al[ok] = (pred[ok] - Pm) @ Rot + Qm
            for rn in REGIONS:
                s2 = ok & reg[rn][:, None]
                row[f'{mode}_{rn}'] = round(rmsd(al[s2], ref[s2]), 3) if s2.sum() >= 3 else None
        rows.append(row)
    except Exception as e:
        R.setdefault('score_err', []).append(f'{tid}:{type(e).__name__}:{str(e)[:90]}')

R['rows'] = rows
lost = len(TG) - len(rows)
print(f'scored {len(rows)} / {len(TG)}  (unscored {lost})')
for k in ('missing','seq_mismatch','skipped','score_err'):
    if R.get(k): print(f'  {k}: {len(R[k])}  e.g. {R[k][:3]}')
assert lost < 0.05 * len(TG), f'{lost} unscored -- too many to trust the medians'

import csv
cols = (['instance','tid','L','cdr3_len','n_atoms_matched','seq_offset','pred_len',
         'method','holo','ab_split','resolution']
        + [f'{m}_{r}' for m in ('chain','fw') for r in REGIONS])
json.dump(rows, open(f'{WORK}/per_target_scores_nbb2.json','w'), indent=1)
with open(f'{WORK}/per_target_scores_nbb2.csv','w',newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=cols, extrasaction='ignore', restval='')
    w.writeheader(); w.writerows(rows)
shutil.rmtree(PRED, ignore_errors=True)
note('scored')

## Results — paired against SimpleFold on identical targets

In [ ]:
def summarise(rr):
    rng = np.random.default_rng(0)
    out = {'n': len(rr)}
    for mode in ('chain','fw'):
        d = {}
        for rn in REGIONS:
            v = np.array([r[f'{mode}_{rn}'] for r in rr if r.get(f'{mode}_{rn}') is not None])
            if len(v) >= 5:
                bs = np.median(v[rng.integers(0, len(v), (2000, len(v)))], axis=1)
                d[rn] = {'median': round(float(np.median(v)), 3), 'mean': round(float(v.mean()), 3),
                         'ci95': [round(float(np.percentile(bs, 2.5)), 3),
                                  round(float(np.percentile(bs, 97.5)), 3)], 'n': int(len(v))}
        out[f'{mode}_aligned'] = d
    return out

S = summarise(rows)
sf = R['reference_only']['SimpleFold_100M_this_benchmark']
ca = S['chain_aligned']
print(f"chain-aligned, n={S['n']}\n")
print(f"{'region':<11}{'NBB2':>9}{'95% CI':>18}{'SimpleFold':>12}{'delta':>9}")
for rn in REGIONS:
    if rn not in ca: continue
    e = ca[rn]
    ci = '[' + str(e['ci95'][0]) + ', ' + str(e['ci95'][1]) + ']'
    print(f"{rn:<11}{e['median']:>9.3f}{ci:>18}{sf[rn]:>12.3f}{e['median']-sf[rn]:>+9.3f}")
print('\nSame 597 targets, same scoring code -- so this delta IS a controlled comparison.')
print('fw-aligned:', {rn: S['fw_aligned'][rn]['median'] for rn in REGIONS if rn in S['fw_aligned']})

strata = {'ab_split==test': [r for r in rows if r['ab_split']=='test'],
          'method==XRAY':   [r for r in rows if r['method']=='XRAY'],
          'method==EM':     [r for r in rows if r['method']=='EM'],
          'holo==True':     [r for r in rows if str(r['holo'])=='True'],
          'holo==False':    [r for r in rows if str(r['holo'])=='False']}
for lo, hi in ((0,13),(14,17),(18,99)):
    strata[f'cdr3_len {lo}-{hi}'] = [r for r in rows if lo <= r['cdr3_len'] <= hi]
S['strata'] = {}
print(f"\n{'stratum':<18}{'n':>5}{'cdr3 median':>13}")
for k, rr in strata.items():
    if len(rr) < 5: continue
    ss = summarise(rr); e = ss['chain_aligned'].get('cdr3')
    if not e: continue
    S['strata'][k] = ss
    print(f"{k:<18}{len(rr):>5}{e['median']:>13.3f}")
R['summary'] = S
note('done')
print('\nwrote results_nbb2.json, per_target_scores_nbb2.{json,csv}')